# G-FOLD: Fuel-Optimal Powered Descent Guidance via Lossless Convexification
## Full 3-DoF (3-Dimensional Degrees of Freedom) Derivation and CVXPY Implementation

**References:**
1. Açıkmeşe, B., & Ploen, S. R. (2007). *Convex programming approach to powered descent guidance for Mars landing*. Journal of Guidance, Control, and Dynamics, 30(5), 1353-1366.
2. Açıkmeşe, B., Carson, J. M., & Blackmore, L. (2013). *Lossless convexification of nonconvex control bound and pointing constraints of the soft landing optimal control problem*. IEEE Transactions on Control Systems Technology, 21(6), 2104-2113.
3. Blackmore, L., Açıkmeşe, B., & Scharf, D. P. (2010). *Minimum-landing-error powered-descent guidance for Mars landing using convex optimization*. Journal of Guidance, Control, and Dynamics, 33(4), 1161-1171.

## 1. What is 3-DoF in Powered Descent Guidance?

In aerospace flight mechanics:
- **1-DoF (1 Degree of Freedom):** Pure vertical flight along the $z$-axis ($z, v_z$). No horizontal divert capability.
- **2-DoF (2 Degrees of Freedom):** Planar flight ($x, z, v_x, v_z$). Treats crossrange as zero.
- **3-DoF (3 Degrees of Freedom):** Full 3D spatial point-mass translational dynamics:
  - **3D Position:** $\mathbf{r}(t) = [x(t), y(t), z(t)]^\top \in \mathbb{R}^3$ ($x$: downrange, $y$: crossrange, $z$: altitude above the landing pad).
  - **3D Velocity:** $\mathbf{v}(t) = [v_x(t), v_y(t), v_z(t)]^\top \in \mathbb{R}^3$.
  - **3D Thrust Vector:** $\mathbf{u}(t) = [u_x(t), u_y(t), u_z(t)]^\top \in \mathbb{R}^3$ vectored in any 3D spherical direction within the engine gimbal cone.

*(Note: 6-DoF adds the 3 rotational degrees of freedom: roll, pitch, yaw attitude and body angular rates $\boldsymbol{\omega}$, coupled via moments of inertia. In guidance architectures like SpaceX Falcon 9 and NASA Mars landers, 3-DoF guidance generates the optimal reference trajectory and thrust acceleration vector $\mathbf{u}^*(t)$, while the inner-loop 6-DoF attitude controller commands the engine gimbal actuators and cold gas thrusters to track the guidance vector.)*

## 2. Mathematical Formulation in 3-DoF

### A. Equations of Motion
$$\dot{\mathbf{r}}(t) = \mathbf{v}(t)$$
$$\dot{\mathbf{v}}(t) = \mathbf{g} + \mathbf{u}(t) + \mathbf{a}_{\text{drag}}(t) + \mathbf{w}(t)$$
where $\mathbf{g} = [0, 0, -g]^\top$, $\mathbf{w}(t) = [w_x, w_y, 0]^\top$ is the 3D wind field, and aerodynamic drag is modeled as:
$$\mathbf{a}_{\text{drag}} = -\frac{1}{2m} \rho(z) C_D A \|\mathbf{v} - \mathbf{w}\| (\mathbf{v} - \mathbf{w})$$

### B. Boundary Conditions
$$\mathbf{r}(0) = \mathbf{r}_0, \quad \mathbf{v}(0) = \mathbf{v}_0, \quad \mathbf{r}(t_f) = \mathbf{0}_{3\times 1}, \quad \mathbf{v}(t_f) = \mathbf{0}_{3\times 1}$$

### C. The 3D Second-Order Cone Constraints
1. **3D Thrust Magnitude Slack (Lossless Convexification):**
   $$\|\mathbf{u}(t)\|_2 = \sqrt{u_x(t)^2 + u_y(t)^2 + u_z(t)^2} \le \Gamma(t)$$
2. **Throttle Box Bounds:**
   $$u_{\min} \le \Gamma(t) \le u_{\max}$$
3. **3D Gimbal Pointing Cone (Thrust Angle Limit):**
   The angle between the thrust vector and the vertical body axis $\hat{\mathbf{e}}_z = [0, 0, 1]^\top$ must satisfy:
   $$\mathbf{u}(t)^\top \hat{\mathbf{e}}_z = u_z(t) \ge \Gamma(t) \cos \theta_{\max}$$
4. **3D Glide-Slope Cone:**
   The spacecraft must remain inside an inverted cone centered on the landing pad to avoid terrain obstacles:
   $$z(t) \ge \sqrt{x(t)^2 + y(t)^2} \tan \gamma_{\text{gs}}$$
5. **Objective Function:**
   $$\min_{\mathbf{u}, \Gamma} \int_0^{t_f} \Gamma(t) \, dt$$

In [ ]:
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

print("CVXPY Version:", cp.__version__)
print("Installed Solvers:", cp.installed_solvers())


In [ ]:
def solve_gfold_3d(
    r0=np.array([200.0, 150.0, 500.0]),   # [x0 (downrange), y0 (crossrange), z0 (altitude)]
    v0=np.array([12.0, -8.0, -40.0]),     # [vx0, vy0, vz0]
    tf=12.0,
    N=60,
    g=9.80665,
    a_max=25.0,
    a_min_ratio=0.3,
    theta_max_deg=35.0,
    gamma_gs_deg=20.0,
    wind=np.array([10.0, -5.0, 0.0])
):
    dt = tf / N
    a_min = a_max * a_min_ratio
    cos_theta = np.cos(np.radians(theta_max_deg))
    tan_gamma = np.tan(np.radians(gamma_gs_deg))
    g_vec = np.array([0.0, 0.0, -g])

    # 3D Variables
    r = cp.Variable((3, N + 1))
    v = cp.Variable((3, N + 1))
    u = cp.Variable((3, N))
    gamma = cp.Variable(N)

    cost = cp.sum(gamma) * dt

    constraints = [
        r[:, 0] == r0, v[:, 0] == v0,
        r[:, N] == np.zeros(3), v[:, N] == np.zeros(3)
    ]

    for k in range(N):
        a_net = u[:, k] + g_vec
        constraints += [
            r[:, k+1] == r[:, k] + v[:, k] * dt + 0.5 * a_net * (dt**2),
            v[:, k+1] == v[:, k] + a_net * dt,

            # 3D Second-Order Cone Constraint (Lossless Convexification):
            cp.norm(u[:, k], 2) <= gamma[k],
            gamma[k] >= a_min,
            gamma[k] <= a_max,

            # 3D Gimbal Pointing Cone:
            u[2, k] >= gamma[k] * cos_theta,

            # 3D Glide-Slope Cone:
            r[2, k] >= cp.norm(r[0:2, k], 2) * tan_gamma,
            r[2, k] >= 0.0
        ]

    prob = cp.Problem(cp.Minimize(cost), constraints)
    prob.solve(solver=cp.CLARABEL if 'CLARABEL' in cp.installed_solvers() else cp.ECOS)

    u_norms = np.linalg.norm(u.value, axis=0)
    slack_gap = np.max(np.abs(gamma.value - u_norms))
    
    return {
        "status": prob.status,
        "t": np.linspace(0, tf, N + 1),
        "x": r.value[0, :], "y": r.value[1, :], "z": r.value[2, :],
        "vx": v.value[0, :], "vy": v.value[1, :], "vz": v.value[2, :],
        "ux": u.value[0, :], "uy": u.value[1, :], "uz": u.value[2, :],
        "gamma": gamma.value, "u_norm": u_norms,
        "slack_gap": slack_gap,
        "fuel_delta_v": prob.value
    }

sol_3d = solve_gfold_3d()
print("3-DoF Status:", sol_3d["status"])
print(f"Total Delta-V: {sol_3d['fuel_delta_v']:.2f} m/s")
print(f"Slack Tightness max |Gamma - ||u|||: {sol_3d['slack_gap']:.2e} m/s^2 (Exact!)")


In [ ]:
# Plotting 3-DoF Trajectory in 3D Space
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

ax.plot(sol_3d["x"], sol_3d["y"], sol_3d["z"], 'b-', lw=3, label="3-DoF Optimal Trajectory")
ax.scatter([sol_3d["x"][0]], [sol_3d["y"][0]], [sol_3d["z"][0]], color='green', s=100, label="Initial Point (r0)")
ax.scatter([0], [0], [0], color='red', marker='x', s=150, lw=3, label="Target Landing Pad (0,0,0)")

# Plot Glide-Slope Cone in 3D
theta_grid = np.linspace(0, 2*np.pi, 30)
z_cone = np.linspace(0, 500, 20)
Theta, Z = np.meshgrid(theta_grid, z_cone)
R_cone = Z / np.tan(np.radians(20))
X_cone = R_cone * np.cos(Theta)
Y_cone = R_cone * np.sin(Theta)
ax.plot_wireframe(X_cone, Y_cone, Z, color='orange', alpha=0.15, label="Glide-Slope Cone (20°)")

ax.set_xlabel("Downrange X [m]")
ax.set_ylabel("Crossrange Y [m]")
ax.set_zlabel("Altitude Z [m]")
ax.set_title("Full 3-DoF G-FOLD Powered Descent Trajectory")
ax.legend()
plt.show()
